In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))

import polars as pl

from IPython.display import Markdown, display

from config import PROCESSED_DATA_DIR
from src.graph.hybrid_features import SAMLDCandidateFeatureBuilder

**Hybrid graph feature preparation and direction-aware rank fusion**

Ring case investigation notebook (08) showed that transaction direction is central to ring recovery. Forward PPR is the strongest primary ranking, reverse PPR recovers fan-in structures, and an equal-weight score average sometimes dilutes a strong one-directional signal

This notebook creates the next modeling layer:
1. build one **label-free feature row** for every non-seed candidate
2. add directional transaction activity and direct exposure to known seeds
3. compare existing score average with **directional maximum** and **reciprocal-rank fusion (RRF)**
4. persist the label-free feature table

Held-out targets and ring identifiers are attached only after feature construction and only for evaluation. The graph nevertheless contains the transactions available to this validation experiment, so the results measure **transductive ring expansion**, not prospective detection of future criminal activity

In [2]:
### open the pooled PPR ring experiment and label-free score table from notebook 07
# ring experiment directory
ring_experiment_dir = PROCESSED_DATA_DIR / 'graph' / 'ring_expansion_validation'

# score table directory
score_table_path = ring_experiment_dir / 'personalized_pagerank_scores.parquet'

# non-seed candidate feature path
candidate_feature_path = ring_experiment_dir / 'candidate_graph_features.parquet'

# create candidate feature builder instance
feature_builder = SAMLDCandidateFeatureBuilder(
    experiment_dir = ring_experiment_dir,
    score_path = score_table_path,
    rrf_constant = 60.0
)

print(
    f'Experiment directory: {ring_experiment_dir}'
)
print(
    f'Feature output: {candidate_feature_path}'
)

Experiment directory: /Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/processed/graph/ring_expansion_validation
Feature output: /Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/processed/graph/ring_expansion_validation/candidate_graph_features.parquet


In [3]:
### build label-free candidate feature table
# candidate feature table
feature_table = feature_builder.build_feature_table(
    force_recompute = False
)


# feature summary
feature_summary = feature_builder.build_feature_summary(
    feature_table = feature_table
)

with pl.Config(
    tbl_cols = -1,
    tbl_rows = -1,
    fmt_str_lengths = 50
):
    display(feature_table.head())
    print(
        f'Feature-table shape: {feature_table.shape}'
    )
    display(feature_summary)

node_id,account,out_degree,in_degree,total_directed_degree,account_transaction_event_count,is_in_seed_weak_component,is_forward_reachable_from_seed,is_reverse_reachable_from_seed,pagerank_unweighted,pagerank_count_weighted,pagerank_log_count_weighted,reverse_pagerank_unweighted,reverse_pagerank_count_weighted,reverse_pagerank_log_count_weighted,bidirectional_pagerank_unweighted,bidirectional_pagerank_count_weighted,bidirectional_pagerank_log_count_weighted,personalized_pagerank_unweighted,personalized_pagerank_count_weighted,personalized_pagerank_log_count_weighted,reverse_personalized_pagerank_unweighted,reverse_personalized_pagerank_count_weighted,reverse_personalized_pagerank_log_count_weighted,bidirectional_personalized_pagerank_unweighted,bidirectional_personalized_pagerank_count_weighted,bidirectional_personalized_pagerank_log_count_weighted,outgoing_transaction_count,outgoing_log_count_weight,incoming_transaction_count,incoming_log_count_weight,forward_seed_neighbor_count,forward_seed_transaction_count,forward_seed_log_count_weight,reverse_seed_neighbor_count,reverse_seed_transaction_count,reverse_seed_log_count_weight,is_direct_forward_seed_neighbor,is_direct_reverse_seed_neighbor,directional_seed_neighbor_count,directional_seed_transaction_count,total_transaction_count_from_edges,directional_max_personalized_pagerank_unweighted,directional_min_personalized_pagerank_unweighted,degree_balance,transaction_count_balance,transaction_events_per_neighbor,is_bidirectional_seed_neighbor,forward_personalized_pagerank_rank,reverse_personalized_pagerank_rank,forward_personalized_pagerank_rank_percentile,reverse_personalized_pagerank_rank_percentile,best_direction_personalized_pagerank_rank,personalized_pagerank_directional_rank_gap,personalized_pagerank_reciprocal_rank_fusion
u64,str,u64,u64,u64,u64,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,u32,f64,f64,bool,bool,u32,f64,f64,f64,f64,f64,f64,f64,bool,u32,u32,f64,f64,u32,i64,f64
0,"""1000005344""",0,1,1,13,false,false,false,9.0945e-7,9.3413e-7,9.2660e-7,4.4813e-7,4.7314e-7,4.6322e-7,6.7879e-7,7.0364e-7,6.9491e-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,2.639057,0,0.0,0.0,0,0.0,0.0,false,false,0,0.0,13.0,0.0,0.0,-1.0,-1.0,13.0,false,7726,4726,0.009905,0.006059,4726,3000,0.000337
1,"""1000008432""",1,0,1,12,false,false,false,8.1270e-7,8.2872e-7,8.2359e-7,0.000001,0.000002,0.000001,0.000001,0.000001,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,2.564949,0.0,0.0,0,0.0,0.0,0,0.0,0.0,false,false,0,0.0,12.0,0.0,0.0,1.0,1.0,12.0,false,7727,4727,0.009906,0.00606,4727,3000,0.000337
2,"""1000038979""",0,1,1,12,false,false,false,0.000001,0.000001,0.000001,4.4813e-7,4.7314e-7,4.6322e-7,7.6200e-7,8.1555e-7,7.9700e-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,2.564949,0,0.0,0.0,0,0.0,0.0,false,false,0,0.0,12.0,0.0,0.0,-1.0,-1.0,12.0,false,7728,4728,0.009908,0.006061,4728,3000,0.000337
3,"""100004626""",1,0,1,12,false,false,false,8.1270e-7,8.2872e-7,8.2359e-7,0.000002,0.000002,0.000002,0.000001,0.000001,0.000001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,2.564949,0.0,0.0,0,0.0,0.0,0,0.0,0.0,false,false,0,0.0,12.0,0.0,0.0,1.0,1.0,12.0,false,7729,4729,0.009909,0.006063,4729,3000,0.000337
4,"""1000065088""",0,1,1,7,false,false,false,8.7572e-7,8.7710e-7,8.8407e-7,4.4813e-7,4.7314e-7,4.6322e-7,6.6192e-7,6.7512e-7,6.7364e-7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,2.079442,0,0.0,0.0,0,0.0,0.0,false,false,0,0.0,7.0,0.0,0.0,-1.0,-1.0,7.0,false,7730,4730,0.00991,0.006064,4730,3000,0.000337


Feature-table shape: (780013, 55)


candidate_count,feature_column_count,forward_reachable_candidate_count,reverse_reachable_candidate_count,direct_forward_seed_neighbor_count,direct_reverse_seed_neighbor_count,bidirectional_seed_neighbor_count
u32,i32,u32,u32,u32,u32,u32
780013,55,7725,4725,3124,1842,428


**Label-free candidate feature table**

The base table reuses graph scores already computed in notebook 07. The *hybrid_features* module adds features that can be calculated using only the observation graph and the known seed set

| Feature family | Examples | Interpretation |
|---|---|---|
| Node activity | degree, transaction-event count, directional transaction count | How active and connected the account is |
| Seed exposure | forward/reverse seed-neighbour counts and transaction counts | Whether the account directly sends to or receives from a known seed |
| Direction balance | degree and transaction-count balance | Whether activity is primarily outgoing or incoming |
| PPR direction | forward, reverse, average, maximum | How seed risk propagates under different edge orientations |
| Rank fusion | deterministic ranks and RRF | Combines rank evidence without averaging raw PPR values |

**Feature-construction note:**

`directional_max_personalized_pagerank_unweighted` preserves the stronger directional score, while RRF combines the forward and reverse ranks as

$$
\operatorname{RRF}(v)
=
\frac{1}{c + r_f(v)}
+
\frac{1}{c + r_r(v)}
$$

where $c=60$ is the smoothing constant, $r_f(v)$ is the forward rank of $v$, and $r_r(v)$ is its reverse rank. A higher RRF value represents stronger combined ranking evidence

**Feature-summary discussion**

- **7,725 candidates** are **forward reachable** from at least one pooled seed, while **4,725** are **reverse reachable**. The graph signal is therefore concentrated in about 1% of the candidate population
- **3,124 candidates** are **direct forward seed neighbors** and **1,842** are **direct reverse seed neighbors**. These represent about **40.4%** and **39.0%** of the respective reachable populations, the remaining reachable candidates require paths longer than one edge
- **428 candidates** are **direct seed neighbors in both direction**

The displayed first 5 rows are ordered by `node_id`, not risk. They are all unreachable from the seeds and have zero PPR scores, so they should not be treated as representative examples

In [4]:
## persist label-free features to designated directory
persisted_feature_path = feature_builder.write_feature_table(
    output_path = candidate_feature_path,
    overwrite = False
)

# validate persisted feature table
feature_builder.validate_feature_file(
    output_path = candidate_feature_path
)

print(
    f'Validated persisted feature table: {persisted_feature_path}'
)

Validated persisted feature table: /Users/gorkemy/Desktop/Projects/Graph-based Financial Crime/data/processed/graph/ring_expansion_validation/candidate_graph_features.parquet


In [5]:
### compare direction-aware ranking rules
# pre-defined ranking scores
ranking_columns = [
    'personalized_pagerank_unweighted',
    'reverse_personalized_pagerank_unweighted',
    'bidirectional_personalized_pagerank_unweighted',
    'directional_max_personalized_pagerank_unweighted',
    'personalized_pagerank_reciprocal_rank_fusion'
]

# ranking evaluation
ranking_evaluation = feature_builder.evaluate_rankings(
    score_columns = ranking_columns,
    k_values = [
        100,
        500,
        1_000,
        5_000,
        10_000
    ],
    feature_table = feature_table
)

with pl.Config(
    tbl_cols = -1,
    tbl_rows = -1,
    fmt_str_lengths = 50
):
    display(ranking_evaluation)

score_name,requested_k,effective_k,candidate_count,target_count,target_hits_at_k,precision_at_k,target_recall_at_k,ring_count,rings_hit_at_k,ring_hit_rate_at_k,first_target_rank
str,i64,i64,i64,i64,i64,f64,f64,i64,i64,f64,i64
"""reverse_personalized_pagerank_unweighted""",100,100,780013,577,12,0.12,0.020797,129,11,0.085271,17
"""bidirectional_personalized_pagerank_unweighted""",100,100,780013,577,12,0.12,0.020797,129,10,0.077519,18
"""directional_max_personalized_pagerank_unweighted""",100,100,780013,577,12,0.12,0.020797,129,10,0.077519,18
"""personalized_pagerank_reciprocal_rank_fusion""",100,100,780013,577,9,0.09,0.015598,129,8,0.062016,34
"""personalized_pagerank_unweighted""",100,100,780013,577,5,0.05,0.008666,129,4,0.031008,12
"""personalized_pagerank_unweighted""",500,500,780013,577,107,0.214,0.185442,129,41,0.317829,12
"""directional_max_personalized_pagerank_unweighted""",500,500,780013,577,42,0.084,0.07279,129,30,0.232558,18
"""personalized_pagerank_reciprocal_rank_fusion""",500,500,780013,577,34,0.068,0.058925,129,24,0.186047,34
"""reverse_personalized_pagerank_unweighted""",500,500,780013,577,29,0.058,0.05026,129,24,0.186047,17


**Direction-aware ranking rules**

The comparison table does not fit a model or tune parameters. It ranks candidates using 5 pre-defined scores and attaches target labels afterward, so each scpre ranks all 708,013 candidates before the 577 target labels are attached:
- forward PPR
- reverse PPR
- existing equal-weight forward/reverse (bidirectional) average
- maximum of the two directional scores
- reciprocal-rank fusion of the two directional ranks

The maximum and RRF alternatives are motivated directly by notebook 08, they allow strong evidence from one direction to survive without being halved by a weak score in the other direction

**Ranking table metrics**

Ranking table uses below metrics as operational trade-offs:
- **precision_at_k**: expected target yield among the **first (k) reviews**
- **target_recall_at_k**: share of all **577** withheld targets recovered
- **rings_hit_at_k**: breadth across **129** approximate rings
- **ring_hit_rate_at_k**: share of rings with at least one recovered target
- **first_target_rank**: how early the first withheld target appears



**What changes with the review budget**

- At **k = 100**, reverse PPR, the average, and directional maximum each recover **12 targets**. Reverse PPR spreads those hits across **11 rings**, compared with **10** for the other two. This is the only evaluated budget where reverse PPR is the strongest early queue
- At **k = 500**, forward PPR recovers **107 targets across 41 rings**. Directional maximum is second with **42 targets across 30 rings**; none of the fusion rules is cose to forward target depth
- At **k = 1_000**, forward PPR recovers **210 targets across 64 rings**. RRF recovers fewer targets but slightly more rings (**156 targets across 66 rings**), while directional maximum produces the widest coverage (**144 targets across 67 rings**)
- At **k = 5_000**, RRF is strongest on target depth, recovering **513 targets across 128 rings**. The average and maximum each reach all **129 rings** but recover about 30 fewer targets
- At **k = 10_000**, the average and maximum recover **547 and 546 targets**, respectively; and all three fusion methods reach **every ring**. Precision fall to roughly **5%**, illustrating the workload cost of the larger queue

The key comparison is at $k = 1,000$ to align with notebook 07 and 08. A fusion method is useful only if it adds target or ring recovery without an unacceptable loss in the other metric. The winner therefore depends n the operational budget. Forward PPR is most efficient around the chosen 1,000 account investigation capacity; whereas combined-direction scores become much more useful when several thousand reviews are available. The strong large-budget performance confirms that forwad and reverse PPR contain complementary target populations; it does not make every fusion rule preferable at a smaller budget

In [6]:
### compare ranking evaluation at an investigation budget k = 1_000
comparison_at_1_000 = (
    ranking_evaluation
    .filter(
        pl.col('requested_k') == 1_000
    )
    .select(
        [
            'score_name',
            'effective_k',
            'target_hits_at_k',
            'precision_at_k',
            'target_recall_at_k',
            'rings_hit_at_k',
            'ring_hit_rate_at_k',
            'first_target_rank'
        ]
    )
    .sort(
        [
            'target_hits_at_k',
            'rings_hit_at_k'
        ],
        descending = [
            True,
            True
        ]
    )
)

with pl.Config(
    tbl_cols = -1,
    tbl_rows = -1,
    fmt_str_lengths = 50
):
    display(comparison_at_1_000)

score_name,effective_k,target_hits_at_k,precision_at_k,target_recall_at_k,rings_hit_at_k,ring_hit_rate_at_k,first_target_rank
str,i64,i64,f64,f64,i64,f64,i64
"""personalized_pagerank_unweighted""",1000,210,0.21,0.363951,64,0.496124,12
"""personalized_pagerank_reciprocal_rank_fusion""",1000,156,0.156,0.270364,66,0.511628,34
"""directional_max_personalized_pagerank_unweighted""",1000,144,0.144,0.249567,67,0.51938,18
"""bidirectional_personalized_pagerank_unweighted""",1000,134,0.134,0.232236,65,0.503876,18
"""reverse_personalized_pagerank_unweighted""",1000,104,0.104,0.180243,60,0.465116,17


**Decision rule for the modeling stage**

- If directional maximum or RRF improves both target hits and ring hits at 1,000, carry it forward as the new unsupervised baseline
- If one improves target depth but the other improves ring breadth, retain both as separate model features rather than declaring one universal winner
- If neither improves on forward PPR, retain forward PPR as the review ranking and still keep reverse PPR as a separate feature; notebook 08 already established that it contains complementary fan-in information

**Decision at the primary investigation budget**

At $k = 1,000$, the trade-off relative to forward PPR is:

| Ranking | Targets | Change vs forward | Rings hit | Change vs forward |
|---|---:|---:|---:|---:|
| Forward PPR | 210 | — | 64 | — |
| RRF | 156 | −54 | 66 | +2 |
| Directional maximum | 144 | −66 | 67 | +3 |
| Equal-weight average | 134 | −76 | 65 | +1 |
| Reverse PPR | 104 | −106 | 60 | −4 |

Forward PPR remains the correct primary review ranking, as its precision is 21.0% and target recall is 36.4% compared with 15.6% / 27.0% for RRF and 14.4% / 25.0 % for directional maximum. RRD sacrifices 25.7% of forward's target hits to reach 2 extra rings, while directional maximum sacrifices 31.4% to reach 3 extra rings

Neither fusion score should replace forward PPR at this investigation budget. Their value is complementary:
- retain RRF as the better depth-breadth compromise and as a downstream feature
- retain directional maximum when ring breadth is prioritized
- keep reverse PPR separately so a model can recognize fan-in behavior rather than forcing both directons into one value

`first_target_rank` only describes the earliest hit and should not drive model choice: forward's first target is rank 12, but its advantage is the much larger number of targets recovered throughout the first 1,000 positions

**RRF limitation**

The current deterministic ranks include candidates with exactly zero PPR. Once the 7,725 positive forward scores or 4,725 positive reverse scores are exhausted, zero-score ties are ordered by `node_id`. RRF consequently gives every candidate a positive value and allows an arbitrary zero-score tie rank to contribute. The current RRF is a valid reproducible baseline, but it should not be tuned on these labels or treated as final evidence. A later robustness test can set the contribution from a zero directional score to zero, or use a tied-rank convention

In [7]:
### construct target feature diagnostics
# attach targets only for feature diagnostics
labeled_evaluation_table = (
    feature_builder.build_labeled_evaluation_table(
        feature_table = feature_table
    )
)

# target feature diagnostics
target_feature_diagnostics = (
    labeled_evaluation_table
    .group_by(
        'is_ring_target'
    )
    .agg(
        [
            # account count
            pl.len()
            .alias(
                'account_count'
            ),
            # median forward ppr
            pl.col('personalized_pagerank_unweighted')
            .median()
            .alias(
                'median_forward_ppr'
            ),
            # median reverse ppr
            pl.col('reverse_personalized_pagerank_unweighted')
            .median()
            .alias(
                'median_reverse_ppr'
            ),
            # median directional max ppr
            pl.col('directional_max_personalized_pagerank_unweighted')
            .median()
            .alias(
                'median_directional_max_ppr'
            ),
            # median RRF
            pl.col('personalized_pagerank_reciprocal_rank_fusion')
            .median()
            .alias(
                'median_rrf'
            ),
            # mean forward seed neighbors
            pl.col('forward_seed_neighbor_count')
            .mean()
            .alias(
                'mean_forward_seed_neighbors'
            ),
            # mean reverse seed neighbors
            pl.col('reverse_seed_neighbor_count')
            .mean()
            .alias(
                'mean_reverse_seed_neighbors'
            ),
            # direct forward seed neighbor share
            pl.col('is_direct_forward_seed_neighbor')
            .mean()
            .alias(
                'direct_forward_seed_neighbor_share'
            ),
            # direct reverse seed neighbor share
            pl.col('is_direct_reverse_seed_neighbor')
            .mean()
            .alias(
                'direct_reverse_seed_neighbor_share'
            )
        ]
    )
    .sort(
        by = 'is_ring_target',
        descending = True
    )
)

with pl.Config(
    tbl_cols = -1,
    tbl_rows = -1,
    fmt_str_lengths = 50
):
    display(target_feature_diagnostics)

is_ring_target,account_count,median_forward_ppr,median_reverse_ppr,median_directional_max_ppr,median_rrf,mean_forward_seed_neighbors,mean_reverse_seed_neighbors,direct_forward_seed_neighbor_share,direct_reverse_seed_neighbor_share
bool,u32,f64,f64,f64,f64,f64,f64,f64,f64
true,577,0.000016,0.0,0.000116,0.001036,0.500867,0.400347,0.487002,0.395147
false,779436,0.0,0.0,0.0,0.000005,0.003648,0.002071,0.003648,0.002071


**Target feature diagnostics**

Two-row target feature diagnostics table asks whether targets have stronger graph signals than the much larger non-target population. Large difference in medians or direct-seed neighbor shares make a feature promising, but they do not establish predictive performance since the labels come from the same validation experiment 

**Discussion on target feature diagnostics**

The target class is extremely rare: **577 of 780,013 candidates**. Even against that imbalance, the label-free graph features show substantial retrospective separation

- **Forward direct exposure**: 48.7% of targets are direct forward seed neighbors versus 0.36% of non-targets, which is almost **133 times** the non-target rate
- **Reverse direct exposure**: 39.5% of targets are direct reverse seed neighbors versus 0.21% of non-targets, which is almost **191 times** the non-target rate
- The target mean neigghbor counts (**0.501 forward** and **0.400 reverse**) are only slightly above their corresponding binary shares. Most directly exposed targets therefore connect to one seed, while a small minority connects to multiple seeds
- Median target forward PPR is **0.000016**, whereas the non-target median is zero. Median reverse PPR is zero at the displayed precision, showing that reverse evidence is strong for a subset rather than the majority of targets
- Directional maximum raises the target median to **0.000116** since it preserves whichever direction carries the stronger signal. Median RRF is **0.001036** for targets versus **0.000005** for non-targets, although the nonzero RRF baseline partly reflects the zero-score rank-ti behavior described above

These differences make directional PPR and direct seed exposure promising hybrid-model inputs, but they are not prospective performance estimates. Approximate targets were reconstructed from suspicious components in the validation period, and the observation graph used for this transductive experiment contains the structural relationships being completed. The large direct-neighbor lifts therefore demonstrate successful **ring-link reconstructon**, not proof that the same features will identify previously unseen future rings

In [8]:
### inspect the first 20 accounts under the two new fusion rules
for score_column in ['directional_max_personalized_pagerank_unweighted', 'personalized_pagerank_reciprocal_rank_fusion']:
    display(
        Markdown(
            f'### Top candidates: {score_column!r}'
        )
    )

    # review queue
    review_queue = (
        labeled_evaluation_table
        .sort(
            [
                score_column,
                'node_id'
            ],
            descending = [
                True,
                False
            ]
        )
        .head(20)
        .select(
            [
                'node_id',
                'account',
                score_column,
                'personalized_pagerank_unweighted',
                'reverse_personalized_pagerank_unweighted',
                'forward_seed_neighbor_count',
                'reverse_seed_neighbor_count',
                'out_degree',
                'in_degree',
                'is_ring_target',
                'ring_id'
            ]
        )
    )

    with pl.Config(
        tbl_cols = -1,
        tbl_rows = -1,
        fmt_str_lengths = 50
    ):
        display(review_queue)

### Top candidates: 'directional_max_personalized_pagerank_unweighted'

node_id,account,directional_max_personalized_pagerank_unweighted,personalized_pagerank_unweighted,reverse_personalized_pagerank_unweighted,forward_seed_neighbor_count,reverse_seed_neighbor_count,out_degree,in_degree,is_ring_target,ring_id
u64,str,f64,f64,f64,u32,u32,u64,u64,bool,u64
775731,"""9946721419""",0.006542,0.000119,0.006542,1,1,1,1,false,null
402277,"""5632446313""",0.005532,0.005532,0.000169,1,1,1,1,false,null
84760,"""1975785639""",0.004573,0.000072,0.004573,1,1,1,1,false,null
383792,"""5420245685""",0.003584,0.000063,0.003584,1,1,1,1,false,null
317146,"""465080959""",0.00355,0.000446,0.00355,1,1,1,1,false,null
589159,"""7793469807""",0.003468,0.000158,0.003468,1,1,1,1,false,null
437679,"""6036399021""",0.003457,0.000446,0.003457,1,1,1,1,false,null
561911,"""7476736900""",0.003451,0.000642,0.003451,1,1,1,1,false,null
212146,"""343937266""",0.003359,0.000342,0.003359,1,1,1,1,false,null


### Top candidates: 'personalized_pagerank_reciprocal_rank_fusion'

node_id,account,personalized_pagerank_reciprocal_rank_fusion,personalized_pagerank_unweighted,reverse_personalized_pagerank_unweighted,forward_seed_neighbor_count,reverse_seed_neighbor_count,out_degree,in_degree,is_ring_target,ring_id
u64,str,f64,f64,f64,u32,u32,u64,u64,bool,u64
561911,"""7476736900""",0.021682,0.000642,0.003451,1,1,1,1,false,null
317146,"""465080959""",0.020109,0.000446,0.00355,1,1,1,1,false,null
437679,"""6036399021""",0.019518,0.000446,0.003457,1,1,1,1,false,null
439528,"""6058065571""",0.018673,0.001145,0.000709,1,1,1,1,false,null
212146,"""343937266""",0.018239,0.000342,0.003359,1,1,1,1,false,null
402277,"""5632446313""",0.017794,0.005532,0.000169,1,1,1,1,false,null
775731,"""9946721419""",0.017415,0.000119,0.006542,1,1,1,1,false,null
589159,"""7793469807""",0.017297,0.000158,0.003468,1,1,1,1,false,null
575616,"""7635866133""",0.017062,0.001418,0.000272,1,1,1,1,false,null


**Review-queue interpretation**

The top-20 directional-maximum queue contains 2 labeled targets, at position 18 and 19, which agrees with its first-target rank of 18. Both targets are reverse-driven; forward PPR is zero, reverse PPR is 0.003209, and each sends directly to one seed. Their larger degree (14 out and 12 in) contrasts with most preceding accounts, which have degree 1-in / 1-out

The top-20 RRF queue contains **no labeled targets**, consistent with its first target appearing at rank 34. RRF strongly favors accounts that rank well in both directions; most of its leading records are direct neighbors of seeds in both directions ad have degree 1-in / 1-out. This explains why RRF is initially weaker but becomes effective at larger review budgets

The accounts marked `is_ring_target = false` are not confirmed legitimate or false positives. They may be normal seed neighbors, suspicious accounts outside the reconstructed target definition, or accounts belonging to patterns not captured by the approximate-ring labels. Their local transactions and seed paths would require investigation before assigning an operational disposition

**Conclusion**

This notebook establishes 4 findings:
1. The persisted **780,013 x 55 candidate feature table** is label-free and suitable as the graph-feature input layer
2. **Forward PPR** remains the primary top-1,000 ranking, providing the best target precision and recall
3. **RRF** and **directional maximum** improve ring breadth, and their advantage becomes substantial at larger review budgets, so they should be retained as complementary features rather than replacements for forward PPR
4. Direct forward and reverse seed exposure strongly separates retrospective targets from non-targets, but the magnitude reflects a transductive ring-expansion setting and must not be presented as future-detection performance